In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import logging
from sklearn.preprocessing import MinMaxScaler
import spectral
import random
import matplotlib.pyplot as plt

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Set a fixed seed for reproducibility
seed = 10
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # If using GPU

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

###########################################
# Helper functions
###########################################
def process_xyz(cube, bands, ill, CMFs):
    """
    Converts hyperspectral cube data to XYZ.
    """
    ill_interp = np.interp(bands, ill[:, 0], ill[:, 1])
    CMFs_interp = np.column_stack([
        np.interp(bands, CMFs[:, 0], CMFs[:, 1]),
        np.interp(bands, CMFs[:, 0], CMFs[:, 2]),
        np.interp(bands, CMFs[:, 0], CMFs[:, 3])
    ])
    sp_tristREF = CMFs_interp * ill_interp[:, None]
    xyz = np.dot(cube, sp_tristREF) / np.sum(sp_tristREF[:, 1], axis=0)
    return xyz

###########################################
# 1. Load spectral data
###########################################
logging.info('Loading spectral data...')
ill = np.loadtxt('../../data/CIE_D65.txt')          
CMFs = np.loadtxt('../../data/CIE2degCMFs_1931.txt')

cube = spectral.open_image('../../data/colorChecker_SG/cubes/cubeCC_120f-ekta100-f14.hdr')
cube_ref = spectral.open_image('../../data/colorChecker_SG/cubeCC_DigitalSG_REF.hdr')

cube_data = cube.load()         
cube_ref_data = cube_ref.load()

wl_input = np.array(cube.metadata['wavelength'], dtype=float)
wl_ref   = np.array(cube_ref.metadata['wavelength'], dtype=float)

###########################################
# 2. Process XYZ data
###########################################
logging.info('Processing XYZ data...')
xyz_input = process_xyz(cube_data, wl_input, ill, CMFs)   
xyz_ref   = process_xyz(cube_ref_data, wl_ref, ill, CMFs)   

###########################################
# 3. Normalize data
###########################################
logging.info('Normalizing data...')
xyz_input_2d = xyz_input.reshape(-1, xyz_input.shape[-1])
xyz_ref_2d   = xyz_ref.reshape(-1, xyz_ref.shape[-1])

scaler_input = MinMaxScaler()
scaler_ref = MinMaxScaler()
X_norm = scaler_input.fit_transform(xyz_input_2d)
Y_norm = scaler_ref.fit_transform(xyz_ref_2d)

X_full = X_norm.reshape(xyz_input.shape)
Y_full = Y_norm.reshape(xyz_ref.shape)

###########################################
# 4. Prepare training data
###########################################
X_flat = X_full.reshape(-1, 3)
Y_flat = Y_full.reshape(-1, 3)

n_pixels = X_flat.shape[0]
train_size = int(0.8 * n_pixels)
train_indices = np.random.choice(n_pixels, train_size, replace=False)
test_indices = np.setdiff1d(np.arange(n_pixels), train_indices)

X_train_split = X_flat[train_indices]
X_test_split  = X_flat[test_indices]
Y_train_split = Y_flat[train_indices]
Y_test_split  = Y_flat[test_indices]

X_train_torch = torch.tensor(X_train_split, dtype=torch.float32)
Y_train_torch = torch.tensor(Y_train_split, dtype=torch.float32)
X_test_torch  = torch.tensor(X_test_split, dtype=torch.float32)
Y_test_torch  = torch.tensor(Y_test_split, dtype=torch.float32)

###########################################
# 5. Define a simple MLP model
###########################################
class SimpleMLP(nn.Module):
    def __init__(self, input_size=3, hidden_size=128, output_size=3):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = SimpleMLP(input_size=3, hidden_size=128, output_size=3)
optimizer = optim.Adam(model.parameters(), lr=0.001)

###########################################
# 6. Angular Loss Function
###########################################
class AngularLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super(AngularLoss, self).__init__()
        self.eps = eps  
    def forward(self, y_pred, y_true):
        y_pred_norm = torch.nn.functional.normalize(y_pred, p=2, dim=1, eps=self.eps)
        y_true_norm = torch.nn.functional.normalize(y_true, p=2, dim=1, eps=self.eps)
        cosine_similarity = torch.sum(y_pred_norm * y_true_norm, dim=1)
        cosine_similarity = torch.clamp(cosine_similarity, -1.0, 1.0)
        angular_loss = 1 - cosine_similarity  
        return angular_loss.mean()

loss_function = AngularLoss()

###########################################
# 7. Training loop (using Angular Loss)
###########################################
epochs = 300
batch_size = 32

logging.info('Training the model...')
for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train_torch.size(0))
    X_train_shuffled = X_train_torch[perm]
    Y_train_shuffled = Y_train_torch[perm]
    
    for i in range(0, X_train_shuffled.size(0), batch_size):
        X_batch = X_train_shuffled[i:i+batch_size]
        Y_batch = Y_train_shuffled[i:i+batch_size]
        
        optimizer.zero_grad()
        Y_pred = model(X_batch)
        loss = loss_function(Y_pred, Y_batch)
        loss.backward()
        optimizer.step()
    
    logging.info(f'Epoch {epoch+1}/{epochs} - Angular Loss: {loss.item()}')

    ###########################################
# 7. Apply correction and evaluate
###########################################
logging.info('Applying correction to the full target...')
X_full_flat = X_full.reshape(-1, 3)
corrected_flat = model(torch.tensor(X_full_flat, dtype=torch.float32)).detach().numpy()
corrected_rgb = scaler_ref.inverse_transform(corrected_flat)
corrected_rgb_image = corrected_rgb.reshape(rgb_ref.shape)

# Compute angular error map
dot_product = np.sum(corrected_rgb * rgb_ref, axis=-1)
norm_pred = np.linalg.norm(corrected_rgb, axis=-1)
norm_true = np.linalg.norm(rgb_ref, axis=-1)
cosine_similarity = dot_product / (norm_pred * norm_true + 1e-6)
angular_error_map = np.arccos(np.clip(cosine_similarity, -1.0, 1.0)) * (180 / np.pi)  # Convert to degrees

# Compute mean and max angular error
mean_error = np.mean(angular_error_map)
max_error = np.max(angular_error_map)
logging.info(f"Mean Angular Error on test set: {mean_error}")
logging.info(f"Max Angular Error on test set: {max_error}")
print("Mean Angular Error:", mean_error)
print("Max Angular Error:", max_error)

# Get train set pixel positions in the image
test_positions = np.unravel_index(test_indices, rgb_ref.shape[:2])

# Plot the Angular Error Map with test set highlights
plt.figure(figsize=(6, 4))
plt.imshow(angular_error_map, cmap='jet')
plt.colorbar(label='Angular Error (degrees)')
plt.scatter(test_positions[1], test_positions[0], s=2, c='white', label='Test Patches', alpha=0.7)
plt.title('Angular Error Map with Test Set Highlighted')
plt.xlabel('Width')
plt.ylabel('Height')
plt.legend(loc='upper right')
plt.show()



2025-02-12 10:42:44,758 - INFO - Loading spectral data...
2025-02-12 10:42:44,760 - INFO - Processing XYZ data...
2025-02-12 10:42:44,760 - INFO - Normalizing data...
2025-02-12 10:42:45,181 - INFO - Training the model...
2025-02-12 10:42:45,188 - INFO - Epoch 1/300 - Angular Loss: 1.6018458604812622
2025-02-12 10:42:45,190 - INFO - Epoch 2/300 - Angular Loss: 1.1726988554000854
2025-02-12 10:42:45,191 - INFO - Epoch 3/300 - Angular Loss: 0.827810525894165
2025-02-12 10:42:45,193 - INFO - Epoch 4/300 - Angular Loss: 0.45486488938331604
2025-02-12 10:42:45,195 - INFO - Epoch 5/300 - Angular Loss: 0.25676003098487854
2025-02-12 10:42:45,197 - INFO - Epoch 6/300 - Angular Loss: 0.18299254775047302
2025-02-12 10:42:45,199 - INFO - Epoch 7/300 - Angular Loss: 0.12003670632839203
2025-02-12 10:42:45,201 - INFO - Epoch 8/300 - Angular Loss: 0.10830573737621307
2025-02-12 10:42:45,203 - INFO - Epoch 9/300 - Angular Loss: 0.07098475098609924
2025-02-12 10:42:45,205 - INFO - Epoch 10/300 - Angul

NameError: name 'rgb_ref' is not defined